# Jupyter Kernel Subshells

Async kernel supports kernel subshells.

- A subshell is separate instance of a shell and has its own `user_ns` (where objects are stored). 
- The subshell uses the same thead as the main shell.
- The `global_user_ns` is common to all subshells and is the `user_ns` of the main shell.


In [ ]:
import ipylab

app = await ipylab.JupyterFrontEnd().wait_ready()

First, lets open a console for the main shell.

In [ ]:
cc = await app.shell.open_console(activate=False, mode=ipylab.InsertMode.merge_bottom)
await cc.inject("%subshell")

Next, lets create a subshell and open a console for it.

In [ ]:
subshell = app.kernel.subshell_manager.create_subshell()

In [ ]:
ccs = await app.shell.open_console(
    subshell_id=subshell.subshell_id, mode=ipylab.InsertMode.split_right, activate=False, ref=cc
)
await ccs.inject("%subshell")

Let's inject some more code to show that the namespaces differ.

In [ ]:
await cc.inject("a = 1")
await ccs.inject("a = 2")

await cc.inject("a")
await ccs.inject("a")

Here is some more details about the namespace:
- The main shell's `user_ns` and `user_global_ns` are the same object
- A subshell has it's own `user_ns` while the `user_global_ns` is the main shells `user_ns` (which is `user_global_ns`)

In [ ]:
await cc.inject("globals() is locals()")
await ccs.inject("globals() is locals()")

await cc.inject("get_ipython() is app.kernel.main_shell")

await ccs.inject("globals() is app.kernel.main_shell.user_ns")

When we're finished with the subshell, we can stop it.

In [ ]:
cc.close()
ccs.close()

subshell.stop(force=True)
subshell